In [1]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [2]:
finetuned_model_path = "/home/user/Desktop/PROJECT/llama/checkpoint-299"

# 8-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model (CPU only to avoid OOM)
device = "cpu"

print("Loading model on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    finetuned_model_path,
    quantization_config=bnb_config,
    device_map={"": device}
)

model.eval()
print("Model loaded successfully!")


Loading model on CPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

finetuned_model_path = "/home/user/Desktop/PROJECT/llama/checkpoint-299"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    finetuned_model_path,
    quantization_config=bnb_config,
    device_map={"": device}
)
model.eval()

print("Model Loaded!")


FileNotFoundError: [Errno 2] No usable temporary directory found in ['/tmp', '/var/tmp', '/usr/tmp', '/home/user/Desktop/PROJECT']

In [7]:
from collections import deque
import torch
import json

conversation_memory = {
    "persona": {
        "name": "Unknown",
        "age": "Unknown",
        "emotion": "neutral",
        "sentiment": "neutral",
        "interests": [],
        "personality_traits": [],
        "key_points": []
    },
    "history": deque(maxlen=10)  # store last 10 messages
}

def update_persona(user_text, memory):
    # Add user message to key points
    memory["persona"]["key_points"].append(user_text)
    
    # Basic emotion detection
    lower = user_text.lower()
    if any(w in lower for w in ["sad", "depressed", "low", "unhappy"]):
        memory["persona"]["emotion"] = "sad"
        memory["persona"]["sentiment"] = "negative"
    elif any(w in lower for w in ["happy", "good", "excited"]):
        memory["persona"]["emotion"] = "happy"
        memory["persona"]["sentiment"] = "positive"
    else:
        memory["persona"]["emotion"] = "neutral"
        memory["persona"]["sentiment"] = "neutral"
    
    return memory

def generate_supporter_response(user_message, memory):
    memory = update_persona(user_message, memory)
    memory["history"].append({"user": user_message})

    # Build history string safely
    history_text = "\n".join([f"user: {m['user']}" for m in memory["history"] if "user" in m])

    prompt = f"""### Instruction:
You are the SUPPORTER.

Use the persona and last user message to generate:
1. emotion
2. emotion_stimulus
3. individual_appraisal (2–4 sentences)
4. strategy_reason
5. response

Output JSON only.

### Persona:
{json.dumps(memory['persona'], indent=2)}

### User Message:
{user_message}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )
    response_text = tokenizer.decode(output[0], skip_special_tokens=True)
    memory["history"].append({"supporter": response_text})
    
    return response_text, memory

# -----------------------
# Chat loop
# -----------------------
print("Chat started! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    
    response, conversation_memory = generate_supporter_response(user_input, conversation_memory)
    print("\nSupporter:\n", response)
    
    print("\n--- Current Persona Key Points ---")
    for idx, kp in enumerate(conversation_memory["persona"]["key_points"]):
        print(f"{idx+1}. {kp}")
    print("\n============================\n")


Chat started! Type 'exit' to quit.


Supporter:
 ### Instruction:
You are the SUPPORTER.

Use the persona and last user message to generate:
1. emotion
2. emotion_stimulus
3. individual_appraisal (2–4 sentences)
4. strategy_reason
5. response

Output JSON only.

### Persona:
{
  "name": "Unknown",
  "age": "Unknown",
  "emotion": "sad",
  "sentiment": "negative",
  "interests": [],
  "personality_traits": [],
  "key_points": [
    "hi, i am feeling low"
  ]
}

### User Message:
hi, i am feeling low

### Response:
I see that you're feeling sad lately, is there anything specific that's been bothering you? It can be helpful to talk about it if you feel comfortable doing so. Would you like me to listen or offer any advice on how to cope with your current situation?

--- Current Persona Key Points ---
1. hi, i am feeling low



Supporter:
 ### Instruction:
You are the SUPPORTER.

Use the persona and last user message to generate:
1. emotion
2. emotion_stimulus
3. individual_appraisal (2–4 s

In [5]:
user_input = "I am feeling really lonely and stressed."
response, conversation_memory = generate_supporter_response(user_input, conversation_memory)

print("\n===== Model Output =====\n")
print(response)


/home/user/Desktop/PROJECT/venv/lib/python3.12/site-packages/transformers/generation/utils.py:2532: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(



===== Model Output =====

### Instruction:
You are the SUPPORTER.

Use the persona and last user message to generate:
1. emotion
2. emotion_stimulus
3. individual_appraisal (2–4 sentences)
4. strategy_reason
5. response

Output JSON only.

### Persona:
{
  "name": "Unknown",
  "age": "Unknown",
  "emotion": "neutral",
  "sentiment": "neutral",
  "interests": [],
  "personality_traits": [],
  "key_points": [
    "I am feeling really lonely and stressed."
  ]
}

### User Message:
I am feeling really lonely and stressed.

### Response:
It sounds like you're experiencing a lot of stress right now, especially with loneliness. It can be tough when things aren't going well in our personal lives or when we don't feel connected with others. I want you to know that it takes courage to reach out for help and support during these times. Have there been any specific situations where this has happened recently?
